In [22]:
import awkward as ak
import numpy as np
import uproot as uproot
import matplotlib.pyplot as plt
import mplhep as hep
import vector as vec

import matplotlib

from numba import prange

plt.style.use(hep.style.CMS)
%matplotlib inline

In [2]:
filename='/eos/cms/store/group/dpg_hgcal/comm_hgcal/wredjeb/TICLv5Performance/TimeResolutionPtError0PU/SinglePionTiming_1p9_100GeV/histo/histo_96692_0.root'
file = uproot.open(filename)

In [3]:
file.keys()

['ticlDumper;1',
 'ticlDumper/trackstersCLUE3DHigh;1',
 'ticlDumper/trackstersTiclCandidate;1',
 'ticlDumper/simtrackstersSC;1',
 'ticlDumper/simtrackstersCP;1',
 'ticlDumper/simtracksters2HitsSC;1',
 'ticlDumper/simtracksters2HitsCP;1',
 'ticlDumper/trackstersSuperclusteringDNN;1',
 'ticlDumper/candidates;1',
 'ticlDumper/associations;1',
 'ticlDumper/tracks;1',
 'ticlDumper/simTICLCandidate;1']

In [30]:
alltracksters = file['ticlDumper/trackstersCLUE3DHigh;1']
allsimtrackstersCP = file['ticlDumper/simtrackstersCP;1']
allassociations = file['ticlDumper/associations;1']

In [31]:
simtrackstersCP = allsimtrackstersCP.arrays([
 'regressed_energy',
 'raw_energy',
 'raw_em_energy',
 'raw_pt',
 'raw_em_pt',
 'barycenter_x',
 'barycenter_y',
 'barycenter_z',
 'barycenter_eta',
 'barycenter_phi',
 'EV1',
 'EV2',
 'EV3',
 'eVector0_x',
 'eVector0_y',
 'eVector0_z',
 'sigmaPCA1',
 'sigmaPCA2',
 'sigmaPCA3',
 'regressed_pt',
 'pdgID'])

In [44]:
tracksters = alltracksters.arrays([
 'regressed_energy',
 'raw_energy',
 'raw_em_energy',
 'raw_pt',
 'raw_em_pt',
 'barycenter_x',
 'barycenter_y',
 'barycenter_z',
 'barycenter_eta',
 'barycenter_phi',
 'EV1',
 'EV2',
 'EV3',
 'eVector0_x',
 'eVector0_y',
 'eVector0_z',
 'sigmaPCA1',
 'sigmaPCA2',
 'sigmaPCA3'])

In [45]:
associations = allassociations.arrays(['tsCLUE3D_simToReco_CP',
 'tsCLUE3D_simToReco_CP_score',
 'tsCLUE3D_simToReco_CP_sharedE'])

In [53]:
# for i in prange(len(dumperInput.inputReaders)):
#     dumper = dumperInput.inputReaders[i].ticlDumperReader
#     cands = dumper.candidates
#     simCands = dumper.simCandidates
#     ass = dumper.associations
#     sims = dumper.simTrackstersCP
#     tracks = dumper.tracks
MAX=0.999
for ev in prange(len(tracksters)):
    tsEv = tracksters[ev]
    stsEv = simtrackstersCP[ev]
    assEv = associations[ev]
    print("Event:", ev)
    for stId, simEnergy in enumerate(stsEv.raw_energy):
        score = assEv.tsCLUE3D_simToReco_CP_score[stId]
        simToReco = assEv.tsCLUE3D_simToReco_CP[stId][score<MAX]
        sharedE = assEv.tsCLUE3D_simToReco_CP_sharedE[stId][score<MAX]
        score = score[score<MAX]

        # trackster with the higher shared energy
        refId = simToReco[np.argmax(sharedE)]
        refEta = tsEv.barycenter_eta[refId]
        refPhi = tsEv.barycenter_phi[refId]

        # sort the others by distance in eta-phi
        otherTsEta = tsEv.barycenter_eta
        otherTsPhi = tsEv.barycenter_phi
        distance = ((otherTsEta-refEta)**2 + (otherTsPhi-refPhi)**2)**0.5
        idx_sort = np.array(distance).argsort()
        tsEnergy_sorted=tsEv.raw_energy[idx_sort]

        print(refId, refEta, refPhi, "\n",sorted(distance), distance[idx_sort],tsEnergy_sorted)
        distances=np.linspace(0,1,100)
        


Event: 0
2 -1.8991036415100098 -0.25808918476104736 
 [0.0, 0.023001981899142265, 0.028821192681789398, 0.03044397570192814, 0.03201712295413017, 0.03352043777704239, 0.035655830055475235, 0.05451361835002899, 0.05675478279590607, 0.05755195766687393, 0.05901302024722099, 0.06166969984769821, 0.07156424969434738, 0.07469475269317627, 0.10292714089155197, 4.810998916625977, 4.874884605407715, 4.8802809715271, 4.890256881713867, 4.891257286071777, 4.906425952911377, 4.910660266876221, 4.9164628982543945, 4.923905372619629, 4.924227237701416, 4.9248785972595215, 4.925322532653809, 4.933386325836182, 4.939716339111328, 4.941030025482178, 4.942391872406006, 4.949005603790283, 4.949794292449951, 4.951284885406494, 4.954545021057129, 4.958959579467773, 4.974292755126953, 4.987521648406982, 5.040645122528076] [0, 0.023, 0.0288, 0.0304, 0.032, 0.0335, ... 4.95, 4.95, 4.96, 4.97, 4.99, 5.04] [169, 9.64, 14.8, 1.32, 4.76, 3.59, 5.07, ... 2.72, 3.92, 1.15, 3.07, 0.792, 0.91]
16 1.903093695640564 2